In [0]:
import re
import pyspark.sql.functions as F

# Data Cleansing

In [0]:
df = spark.read.table("fraud_detection_project.bronze_layer.merchant_registry")

def StandardizeNames(df):
    l = df.columns
    cols = []
    for c in l:
        temp = re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower()
        for _ in range(5):
            temp = re.sub(r'\b([a-z])_([a-z])\b', r'\1\2', temp)
            temp = re.sub(r'(?<=[a-z])_([a-z])(?=_|$)', r'\1', temp)
        temp = re.sub(r'_+','_', temp)
        temp = temp.lstrip('_')
        cols.append(temp)
    return df.toDF(*cols)
df = StandardizeNames(df)
df.dtypes

In [0]:
# Deleting duplicated data
df.dropDuplicates(['merchant_id'])

# Deleting rows without some features
df = df.dropna(how='any', subset=['merchant_id','file_path','ingest_datetime'])

In [0]:
# Fill null values with the city name from the address
df = df.withColumn(
    "merchant_city",
    F.when(
        F.col("merchant_city").isNull(),
        F.split(F.col("merchant_full_address"), ",").getItem(2)
    ).otherwise(F.col("merchant_city"))
)

# Replacing "3" for "e
df = df.withColumn(
    "merchant_city",
    F.regexp_replace("merchant_city", "3", "e")
)

# Replacing "@" for "a"
df = df.withColumn(
    "merchant_city",
    F.regexp_replace("merchant_city", "@", "a")
)

# Replacing "1" for "i"
df = df.withColumn(
    "merchant_city",
    F.regexp_replace("merchant_city", "1", "i")
)

# Deleting "_???" from the city name
df = df.withColumn(
    "merchant_city",
    F.regexp_replace("merchant_city", "[_???]", "")
)

df = df.withColumn(
    "merchant_city",
    F.when(
        ~F.col("merchant_city").substr(-1, 1).rlike("[A-Z]"),
        F.concat(
            F.col("merchant_city"),
            F.upper(
                F.right(
                F.split(F.col("merchant_full_address"), ",").getItem(2).cast("string"), 
                F.lit(5)
            )
            )
        )
    ).otherwise(F.col("merchant_city"))
)

# Remove extra spaces and adjust the captalization (E.g: "da paz" -> "Da Paz")
df = df.withColumn("merchant_city", F.trim(F.initcap(F.col("merchant_city")))
)


In [0]:
target = "fraud_detection_project.silver_layer.merchant_registry"
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)

In [0]:
%sql
select * FROM fraud_detection_project.silver_layer.merchant_registry LIMIT(200)